# Goal-Directed Folding — Simulation Evaluation

Companion to `alignment_sim.ipynb`, but for the **goal-directed folding** task
(`canonicalisation_alignment_centre_sleeve_folding`, $K=3$ ordered sub-goals: flattened $S_0$,
fold-1 $S_1$, fold-2 $S_2$; 25-step horizon, no early stopping).

It produces the two paper deliverables:
1. the **quantitative table** (`\label{tab:sim-folding}`) comparing MAGPIE variants + Human, and
2. **trajectory filmstrips** (`\label{fig:qualitative_folding}`).

**Folding metrics** (from `performance.csv`, all reported at the final trajectory step):

| Metric | Column | Direction | Meaning |
|---|---|---|---|
| MPD | `evaluation/mean_particle_distance` | $\downarrow$ | mean per-particle L2 to goal after rigid Procrustes alignment |
| MKD | `evaluation/semantic_keypoint_distance` | $\downarrow$ | same, over semantic keypoints only |
| A-IoU | `evaluation/algn_IoU` | $\uparrow$ | direct mask overlap with the target fold (no re-alignment) |
| Max IoU | `evaluation/max_IoU` | $\uparrow$ | mask overlap maximised over a rigid 2D rotation + translation |
| NC | `evaluation/normalised_coverage` | $\uparrow$ | coverage / flattened coverage |
| Sq-IoU-SR | `evaluation/success` | $\uparrow$ | sequential subgoal success (already latched in-engine per step) |

Unlike the alignment notebook, **success is not re-derived from IoU+NC thresholds**: the folding
engine already computes the sequential IoU-based success (`success()` in `garment_folding.py`) and
stores it per step in `evaluation/success`. We only aggregate it (any-step vs last-step).

> The evaluation data lives on the compute cluster. Point `DATA_DIR` at the run root; any experiment
> whose `performance.csv` is missing falls back to reproducible dummy numbers so the notebook still
> renders end-to-end. The local `../../tmp` run (human folding) is wired up as a working example.

In [ ]:
import os
import ast
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.lines import Line2D
from IPython.display import display

# ---- Publication-ready style (matches alignment_sim.ipynb) ----
plt.rcParams.update({
    'font.size': 18, 'axes.labelsize': 18, 'axes.titlesize': 18,
    'xtick.labelsize': 16, 'ytick.labelsize': 16, 'legend.fontsize': 15,
    'font.family': 'sans-serif', 'lines.linewidth': 2.5, 'axes.linewidth': 1.5,
    'xtick.major.width': 1.5, 'ytick.major.width': 1.5,
})

# One stable colour per folding method (paper column order).
COLORS = {
    'MAGPIE (All-Garment)': '#d62728',
    'MAGPIE (LS-only)':     '#ff7f0e',
    'MAGPIE + Heuristic':   '#1f77b4',
    'Human':                '#2ca02c',
}


def parse_list(val, clip=True):
    """Parse a CSV cell holding a python-list string into a list of floats.

    clip=True clamps to [0, 1] (for IoU / NC / success flags); clip=False keeps raw
    values (for distances such as MPD / MKD, which are NOT bounded by 1).
    """
    if pd.isna(val):
        return []
    if isinstance(val, list):
        raw = val
    elif isinstance(val, str):
        try:
            parsed = ast.literal_eval(val)
            raw = parsed if isinstance(parsed, list) else [parsed]
        except (ValueError, SyntaxError):
            try:
                raw = [float(val)]
            except ValueError:
                raw = []
    elif isinstance(val, (int, float)):
        raw = [val]
    else:
        raw = []
    out = []
    for x in raw:
        try:
            x = float(x)
        except (ValueError, TypeError):
            continue
        if clip:
            x = max(0.0, min(1.0, x))
        out.append(x)
    return out


print('Setup complete.')

In [ ]:
# =====================================================================================
#  Folding aggregation
# =====================================================================================
# Column map: (key -> csv column, clip_to_unit_interval)
FOLD_COLS = {
    'mpd':    ('evaluation/mean_particle_distance',     False),  # lower better
    'mkd':    ('evaluation/semantic_keypoint_distance', False),  # lower better
    'aiou':   ('evaluation/algn_IoU',                   True),   # higher better
    'maxiou': ('evaluation/max_IoU',                    True),   # higher better
    'nc':     ('evaluation/normalised_coverage',        True),   # higher better
    'succ':   ('evaluation/success',                    True),   # 0/1 latched flag
}
# Metrics where a smaller value is better (used for 'best-so-far' and table ranking).
LOWER_IS_BETTER = {'mpd', 'mkd'}


def aggregate_folding_performance(csv_path, max_steps=25):
    """Aggregate one experiment's performance.csv into folding summary statistics.

    For each trajectory the series is truncated to the first `max_steps` steps, then:
      * final value  = value at the last observed step,
      * best value   = min (distances) or max (IoU/NC) over the trajectory,
      * Sq-IoU-SR    = any-step (max of the latched success flag) and last-step success.
    Distances (MPD/MKD) are parsed unclipped; IoU/NC/success are clamped to [0, 1].
    """
    df = pd.read_csv(csv_path)
    series = {k: df[col].apply(lambda v, c=clip: parse_list(v, clip=c))
              for k, (col, clip) in FOLD_COLS.items() if col in df.columns}

    n_traj = len(df)
    acc = {k: {'final': [], 'best': []} for k in FOLD_COLS}
    any_succ, last_succ = [], []
    step_curves = {k: [] for k in FOLD_COLS}  # per-trajectory truncated series, for curves

    for i in range(n_traj):
        succ = series.get('succ', pd.Series([[]]*n_traj)).iloc[i][:max_steps]
        any_succ.append(1 if (len(succ) and max(succ) >= 0.5) else 0)
        last_succ.append(1 if (len(succ) and succ[-1] >= 0.5) else 0)
        for k in FOLD_COLS:
            if k not in series:
                continue
            vals = series[k].iloc[i][:max_steps]
            step_curves[k].append(vals)
            if not vals:
                continue
            acc[k]['final'].append(vals[-1])
            acc[k]['best'].append(min(vals) if k in LOWER_IS_BETTER else max(vals))

    out = {'Total_traj': n_traj,
           'SqSR_any_successes': int(sum(any_succ)),
           'SqSR_last_successes': int(sum(last_succ)),
           'SqSR_any_mean': float(np.mean(any_succ)) if any_succ else 0.0,
           'SqSR_last_mean': float(np.mean(last_succ)) if last_succ else 0.0}
    for k in FOLD_COLS:
        for which in ('final', 'best'):
            arr = acc[k][which]
            out[f'{k}_{which}_mean'] = float(np.mean(arr)) if arr else np.nan
            out[f'{k}_{which}_std'] = float(np.std(arr)) if arr else np.nan
    out['_curves'] = step_curves
    return out


def _dummy_folding_metrics(seed_label):
    """Reproducible placeholder numbers so the notebook renders without cluster data."""
    rng = np.random.default_rng(abs(hash(seed_label)) % (2**32))
    q = rng.uniform(0.55, 0.9)  # overall competence knob for this fake method
    out = {'Total_traj': 30,
           'SqSR_any_successes': int(round(30 * min(1, q + 0.1))),
           'SqSR_last_successes': int(round(30 * q)),
           'SqSR_any_mean': min(1.0, q + 0.1), 'SqSR_last_mean': q}
    for k, hi in [('aiou', 0.9), ('maxiou', 0.95), ('nc', 0.97)]:
        for which in ('final', 'best'):
            out[f'{k}_{which}_mean'] = min(hi, q + (0.05 if which == 'best' else 0.0))
            out[f'{k}_{which}_std'] = rng.uniform(0.03, 0.09)
    for k in ('mpd', 'mkd'):
        for which in ('final', 'best'):
            out[f'{k}_{which}_mean'] = (1 - q) * 0.6 + (0.0 if which == 'best' else 0.03)
            out[f'{k}_{which}_std'] = rng.uniform(0.01, 0.05)
    out['_curves'] = {}
    return out


def load_metrics(data_dir, exp, max_steps=25, use_dummy=True):
    """Load metrics for one experiment dict, falling back to dummy data if missing."""
    path = os.path.join(data_dir, exp['name'], exp['check'], 'performance.csv')
    if os.path.exists(path):
        m = aggregate_folding_performance(path, max_steps=max_steps)
        m['_source'] = 'real'
        return m
    if use_dummy:
        print(f"[dummy] {exp['label']}: not found -> {path}")
        m = _dummy_folding_metrics(exp['label'])
        m['_source'] = 'dummy'
        return m
    print(f"[skip]  {exp['label']}: not found -> {path}")
    return None


print('Aggregation helpers ready.')

In [ ]:
# =====================================================================================
#  Experiment definitions (edit paths to match your run root)
# =====================================================================================
# On the cluster the folding runs live under the same save_root as the alignment runs,
# e.g. "/users/hcv530/scratch/garment_folding_data". Locally, the human folding run is
# written into the repo's ./tmp by the current evaluation job.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
LOCAL_TMP = os.path.join(REPO_ROOT, 'tmp')
CLUSTER_DIR = '/users/hcv530/scratch/garment_folding_data'

# Use the cluster root if present, else fall back to the local ./tmp (has the human run).
DATA_DIR = CLUSTER_DIR if os.path.isdir(CLUSTER_DIR) else LOCAL_TMP
MAX_STEPS = 25   # folding horizon (arena ...workspace_horizon_25)
print('DATA_DIR =', DATA_DIR)

MAGPIE_EXP = 'magpie_ctr_align_all_sim_garments_p4_v126_hindsight'
FOLD_ARENA = 'multi_longsleeve_provide_semkey_pixel_no_success_stop_resol_128_workspace_horizon_25'
FOLD_TASK = 'canonicalisation_alignment_centre_sleeve_folding'

# Paper table columns (tab:sim-folding): three MAGPIE variants + Human.
folding_experiments = [
    {'name': f'transfer_eval/{MAGPIE_EXP}/{FOLD_ARENA}/{FOLD_TASK}',
     'check': 'eval_checkpoint_-2', 'label': 'MAGPIE (All-Garment)'},
    {'name': 'magpie_canon_align_folding_longsleeve_p4_v80_no_state',
     'check': 'eval_checkpoint_-2', 'label': 'MAGPIE (LS-only)'},
    {'name': 'magpie_heuristic_folding_longsleeve_through_canon_alignment',
     'check': 'eval_checkpoint_-2', 'label': 'MAGPIE + Heuristic'},
    {'name': 'human_longsleeve_folding_through_canon_alignment',
     'check': 'eval_checkpoint_-2', 'label': 'Human'},
]

# Load everything once and reuse.
METRICS = {e['label']: load_metrics(DATA_DIR, e, max_steps=MAX_STEPS) for e in folding_experiments}
print('\nSources:', {k: v['_source'] for k, v in METRICS.items()})

In [ ]:
# =====================================================================================
#  LaTeX table  ->  drop-in body for \label{tab:sim-folding}
# =====================================================================================
# Rows follow the paper: MPD (down), A-IoU, Max IoU, Sq-IoU-SR. MKD is included as an
# optional extra row (set INCLUDE_MKD=False to match the current paper table exactly).
INCLUDE_MKD = True
REPORT = 'final'   # 'final' (paper: reported at final step) or 'best' (best-so-far)


def _fmt_pct(mean, std):
    if mean is None or (isinstance(mean, float) and np.isnan(mean)):
        return '--'
    return f"{mean*100:.1f} $\\pm$ {std*100:.1f}"


def _fmt_dist(mean, std):
    if mean is None or (isinstance(mean, float) and np.isnan(mean)):
        return '--'
    return f"{mean:.3f} $\\pm$ {std:.3f}"


def generate_folding_latex_table(metrics_by_label, labels, report='final',
                                 include_mkd=True, sr_kind='last'):
    """Build the tabular body for tab:sim-folding with best-cell highlighting.

    sr_kind: 'last' (holding the final fold at horizon end) or 'any' (reached at some step).
    """
    # (row label, metric key, kind) — kind in {'pct','dist','sr'}
    rows = [('MPD $\\downarrow$',    'mpd',    'dist')]
    if include_mkd:
        rows.append(('MKD $\\downarrow$', 'mkd', 'dist'))
    rows += [('A-IoU $\\uparrow$',   'aiou',   'pct'),
             ('Max IoU $\\uparrow$', 'maxiou', 'pct'),
             ('Sq-IoU-SR $\\uparrow$', 'succ',  'sr')]

    # ---- gather numeric values + display strings ----
    numeric = {r[0]: {} for r in rows}
    display_s = {r[0]: {} for r in rows}
    for name, key, kind in rows:
        for lab in labels:
            m = metrics_by_label.get(lab)
            if m is None:
                numeric[name][lab], display_s[name][lab] = None, '--'
                continue
            if kind == 'sr':
                succ = m[f'SqSR_{sr_kind}_successes']
                tot = m['Total_traj']
                numeric[name][lab] = m[f'SqSR_{sr_kind}_mean']
                display_s[name][lab] = f"{succ}/{tot}"
            elif kind == 'dist':
                mean, std = m[f'{key}_{report}_mean'], m[f'{key}_{report}_std']
                numeric[name][lab] = mean
                display_s[name][lab] = _fmt_dist(mean, std)
            else:  # pct
                mean, std = m[f'{key}_{report}_mean'], m[f'{key}_{report}_std']
                numeric[name][lab] = mean
                display_s[name][lab] = _fmt_pct(mean, std)

    # ---- rank + colour the best cell per row (green best, yellow 2nd, red 3rd) ----
    for name, key, kind in rows:
        lower = key in LOWER_IS_BETTER
        vals = [(v, lab) for lab, v in numeric[name].items() if v is not None and not (isinstance(v, float) and np.isnan(v))]
        if not vals:
            continue
        ordered = sorted({v for v, _ in vals}, reverse=not lower)
        shade = {0: 'green!25', 1: 'yellow!25', 2: 'red!25'}
        for v, lab in vals:
            rank = ordered.index(v)
            if rank in shade:
                display_s[name][lab] = f"\\cellcolor{{{shade[rank]}}}" + display_s[name][lab]

    # ---- emit tabular (mirrors the l|ccc||c layout in example.tex) ----
    magpie_cols = [l for l in labels if l != 'Human']
    n_mag = len(magpie_cols)
    lines = []
    lines.append('\\begin{tabular}{l|' + 'c' * n_mag + '||c}')
    lines.append('\\toprule')
    lines.append(f' & \\multicolumn{{{n_mag}}}{{c||}}{{\\textbf{{MAGPIE}}}} & \\\\')
    lines.append(f'\\cmidrule(lr){{2-{n_mag+1}}}')
    short = {'MAGPIE (All-Garment)': '\\textbf{All-Garment}',
             'MAGPIE (LS-only)': '\\textbf{LS-only}',
             'MAGPIE + Heuristic': '\\textbf{+\\,Heuristic}',
             'Human': '\\textbf{Human}'}
    header = ['\\textbf{Metric}'] + [short.get(l, l) for l in magpie_cols + ['Human']]
    lines.append(' & '.join(header) + ' \\\\')
    lines.append('\\midrule')
    for name, key, kind in rows:
        cells = [name] + [display_s[name][l] for l in magpie_cols + ['Human']]
        lines.append(' & '.join(cells) + ' \\\\')
    lines.append('\\bottomrule')
    lines.append('\\end{tabular}')
    return '\n'.join(lines)


labels = [e['label'] for e in folding_experiments]
latex = generate_folding_latex_table(METRICS, labels, report=REPORT,
                                     include_mkd=INCLUDE_MKD, sr_kind='last')
print('% ---- paste into example.tex (tab:sim-folding) ----')
print(latex)

In [ ]:
# =====================================================================================
#  Summary bar chart (final-step folding metrics per method)
# =====================================================================================
def plot_folding_summary(metrics_by_label, labels, report='final', sr_kind='last'):
    panels = [
        ('Sq-IoU-SR', 'succ', True),      # success rate, higher better
        ('A-IoU',     'aiou', True),
        ('Max IoU',   'maxiou', True),
        ('MPD',       'mpd', False),      # distance, lower better
    ]
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    x = np.arange(len(labels))
    for ax, (title, key, higher) in zip(axes.flatten(), panels):
        means, stds, colors = [], [], []
        for lab in labels:
            m = metrics_by_label.get(lab)
            colors.append(COLORS.get(lab, '#888888'))
            if m is None:
                means.append(0.0); stds.append(0.0); continue
            if key == 'succ':
                means.append(m[f'SqSR_{sr_kind}_mean']); stds.append(0.0)
            else:
                means.append(m[f'{key}_{report}_mean']); stds.append(m[f'{key}_{report}_std'])
        ax.bar(x, means, yerr=stds, capsize=5, color=colors, edgecolor='black')
        ax.set_title(f"{title} ({'higher' if higher else 'lower'} better)", pad=12)
        ax.set_xticks(x)
        ax.set_xticklabels([l.replace('MAGPIE ', '').replace('(', '').replace(')', '')
                            for l in labels], rotation=15, ha='right')
        ax.grid(axis='y', linestyle='--', alpha=0.6)
        if higher:
            ax.set_ylim(0, 1.05)
    fig.suptitle('Goal-Directed Folding — Long-sleeve (final-step metrics)', fontsize=20, y=1.0)
    plt.tight_layout()
    plt.savefig('folding_summary_bars.png', bbox_inches='tight', dpi=200)
    plt.show()


plot_folding_summary(METRICS, labels, report=REPORT, sr_kind='last')

In [ ]:
# =====================================================================================
#  Per-step progression curves (how the fold assembles over the horizon)
# =====================================================================================
def _mean_curve(curve_list, max_steps):
    """Mean across trajectories at each step, holding the last value once a traj ends."""
    if not curve_list:
        return None
    grid = np.full((len(curve_list), max_steps), np.nan)
    for i, c in enumerate(curve_list):
        for t in range(max_steps):
            grid[i, t] = c[t] if t < len(c) else (c[-1] if c else np.nan)
    return np.nanmean(grid, axis=0)


def plot_folding_curves(metrics_by_label, labels, max_steps=25):
    panels = [('Sq-IoU-SR (fraction succeeded)', 'succ'),
              ('A-IoU to target fold', 'aiou'),
              ('Max IoU to target fold', 'maxiou'),
              ('MPD (lower better)', 'mpd')]
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    steps = np.arange(max_steps)
    any_real = False
    for ax, (title, key) in zip(axes.flatten(), panels):
        for lab in labels:
            m = metrics_by_label.get(lab)
            if m is None or not m.get('_curves'):
                continue
            curve = _mean_curve(m['_curves'].get(key, []), max_steps)
            if curve is None:
                continue
            any_real = True
            ax.plot(steps, curve, label=lab, color=COLORS.get(lab, '#888888'))
        ax.set_title(title, pad=10)
        ax.set_xlabel('step')
        ax.grid(linestyle='--', alpha=0.6)
        if key in ('succ', 'aiou', 'maxiou'):
            ax.set_ylim(0, 1.05)
    handles, lbls = axes[0][0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, lbls, loc='upper center', ncol=len(lbls),
                   bbox_to_anchor=(0.5, 1.04), frameon=False)
    plt.tight_layout()
    if any_real:
        plt.savefig('folding_curves.png', bbox_inches='tight', dpi=200)
        plt.show()
    else:
        plt.close(fig)
        print('No per-step curves available (all experiments used dummy data).')


plot_folding_curves(METRICS, labels, max_steps=MAX_STEPS)

In [ ]:
# =====================================================================================
#  Trajectory filmstrips  ->  figure for \label{fig:qualitative_folding}
# =====================================================================================
# The engine saves each episode as a 6-column grid (episode_<id>_trajectory.png). We
# re-crop it into a single clean filmstrip of hand-picked steps for the paper figure.
def _infer_grid_rows(img, cols=6):
    """Rows in the saved grid, assuming ~square tiles (tiles are 3x3 in matplotlib)."""
    h, w = img.shape[:2]
    return max(1, int(round(h / (w / cols))))


def _cell(img, idx, cols, rows, inset=0.045):
    """Extract grid cell `idx` (row-major) by even division, trimming an `inset` margin."""
    h, w = img.shape[:2]
    ch, cw = h / rows, w / cols
    r, c = idx // cols, idx % cols
    y0 = int((r + inset) * ch); y1 = int((r + 1 - inset) * ch)
    x0 = int((c + inset) * cw); x1 = int((c + 1 - inset) * cw)
    return img[y0:y1, x0:x1]


def plot_folding_trajectories(data_dir, experiments, output_filename,
                              episodes, steps_to_show, cols=6, inset=0.045):
    """One row per method; each row a filmstrip of `steps_to_show` for its chosen episode."""
    n = len(experiments)
    fig, axes = plt.subplots(n, 1, figsize=(2.0 * len(steps_to_show), 2.0 * n))
    if n == 1:
        axes = [axes]
    for ax, exp, ep in zip(axes, experiments, episodes):
        path = os.path.join(data_dir, exp['name'], exp['check'],
                            'performance_visualisation', f'episode_{ep}_trajectory.png')
        if not os.path.exists(path):
            ax.text(0.5, 0.5, f"missing: {exp['label']}\nepisode {ep}",
                    ha='center', va='center', color='gray')
            ax.axis('off'); continue
        img = mpimg.imread(path)
        rows = _infer_grid_rows(img, cols)
        pad_val = 1.0 if np.issubdtype(img.dtype, np.floating) else 255
        tiles, nch = [], (img.shape[2] if img.ndim == 3 else 1)
        for j, s in enumerate(steps_to_show):
            tile = _cell(img, s, cols, rows, inset)
            tiles.append(tile)
            if j < len(steps_to_show) - 1:
                pw = max(2, int(tile.shape[1] * 0.02))
                shape = (tile.shape[0], pw) + ((nch,) if img.ndim == 3 else ())
                tiles.append(np.full(shape, pad_val, dtype=img.dtype))
        strip = np.concatenate([t if t.shape[0] == tiles[0].shape[0]
                                else t[:tiles[0].shape[0]] for t in tiles], axis=1)
        ax.imshow(strip)
        ax.text(strip.shape[1] * 0.005, strip.shape[0] * 0.94, exp['label'],
                color='white', fontsize=13, fontweight='bold', ha='left', va='bottom',
                bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', pad=3))
        ax.axis('off')
    plt.subplots_adjust(hspace=0.05, top=0.98, bottom=0.02, left=0.01, right=0.99)
    plt.savefig(output_filename, bbox_inches='tight', pad_inches=0.05, dpi=300)
    plt.show()
    print('saved ->', output_filename)


# Pick ~9 steps spanning crumpled -> canonicalisation -> fold-1 -> fold-2.
STEPS_TO_SHOW = [0, 2, 4, 6, 8, 12, 16, 20, 24]
EPISODES = [0, 0, 0, 0]  # per-method episode id; tune to a representative rollout each

plot_folding_trajectories(
    DATA_DIR, folding_experiments,
    output_filename='traj_folding_cmp.png',
    episodes=EPISODES, steps_to_show=STEPS_TO_SHOW,
)

In [ ]:
# =====================================================================================
#  Completeness audit (episode counts per folding experiment)
# =====================================================================================
def audit_folding(data_dir, experiments, expected=30):
    rows = []
    for e in experiments:
        path = os.path.join(data_dir, e['name'], e['check'], 'performance.csv')
        if os.path.exists(path):
            try:
                n = len(pd.read_csv(path))
            except Exception:
                n = -1
        else:
            n = 0
        rows.append({'Experiment': e['label'], 'Episodes': n,
                     'Path': os.path.join(e['name'], e['check'])})
    audit = pd.DataFrame(rows).set_index('Experiment')

    def color(v):
        if v == 0:
            return 'background-color:#ffcccc;color:#990000;font-weight:bold;'
        if v == -1:
            return 'background-color:#e6b8af;color:#660000;font-weight:bold;'
        if v < expected:
            return 'background-color:#fff2cc;color:#b26b00;font-weight:bold;'
        return 'background-color:#d9ead3;color:#274e13;font-weight:bold;'

    styled = (audit.style
              .map(color, subset=['Episodes'])
              .set_caption(f"Folding experiment audit (target: {expected} episodes). "
                           "Red=missing, Yellow=incomplete, Green=complete."))
    display(styled)
    return audit


_ = audit_folding(DATA_DIR, folding_experiments, expected=30)